# Fine-tuning NLLB FR↔WO avec MLflow
Entraînement du modèle facebook/nllb-200-distilled-600M sur dataset Wolof-Français.
Tous les métriques, paramètres et le modèle sont loggés dans MLflow.

**Environnement** : Colab GPU (P100 recommandé)

## 0. Installation et configuration

In [17]:
!pip install -q \
    transformers datasets evaluate accelerate peft bitsandbytes \
    sacrebleu mlflow dagshub

import torch
print("CUDA disponible :", torch.cuda.is_available())
print("Device :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA disponible : True
Device : Tesla T4


In [ ]:
# torch.cuda.empty_cache()

# Utiliser le DagsHub client pour se connecter à MLflow

In [21]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
token = user_secrets.get_secret("DAGSHUB_TOKEN")

os.environ["DAGSHUB_USER_TOKEN"] = token

import dagshub
dagshub.init(repo_owner='DrEPL', repo_name='ml-translation-wo-fr', mlflow=True)

Initialized MLflow to track repo "DrEPL/ml-translation-wo-fr"

Repository DrEPL/ml-translation-wo-fr initialized!

## 1. Imports et configuration MLflow

In [20]:
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
import numpy as np
import torch
import os
from datetime import datetime
import json

# MLflow
import mlflow
import mlflow.transformers

# ml_ta_wo_fr
MLFLOW_TRACKING_URI = "https://dagshub.com/DrEPL/ml-translation-wo-fr.mlflow"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print(f"MLflow Tracking URI: {MLFLOW_TRACKING_URI}")
exp = mlflow.get_experiment_by_name("nllb-fr-wo-translation")
if exp is None:
    exp_id = mlflow.create_experiment("nllb-fr-wo-translation")
    print(f"Created experiment with ID: {exp_id}")
else:
    print(f"MLflow Experiment ID: {exp.experiment_id}")

MLflow Tracking URI: https://dagshub.com/DrEPL/ml-translation-wo-fr.mlflow
MLflow Experiment ID: 0


## 2. Chargement et préparation du dataset

In [4]:
# Chargement
dataset_name = "galsenai/centralized_wolof_french_translation_data"
raw_ds = load_dataset(dataset_name)["train"]
raw_ds = raw_ds.rename_columns({"fr": "french", "wo": "wolof"})

print(f"Dataset brut: {len(raw_ds)} exemples")

# Doublement bidirectionnel (FR→WO + WO→FR)
def double_examples(examples):
    sources, targets, tgt_langs = [], [], []
    for fr, wo in zip(examples["french"], examples["wolof"]):
        # FR → WO
        sources.append(fr)
        targets.append(wo)
        tgt_langs.append("wol_Latn")
        # WO → FR
        sources.append(wo)
        targets.append(fr)
        tgt_langs.append("fra_Latn")
    return {"source": sources, "target": targets, "tgt_lang_code": tgt_langs}

bidir_ds = raw_ds.map(
    double_examples,
    batched=True,
    batch_size=2000,
    remove_columns=["french", "wolof", "source"],
    desc="Doublement bidirectionnel"
).shuffle(seed=42)

print(f"Dataset doublé: {len(bidir_ds)} exemples")

# Split 88 / 6 / 6
split = bidir_ds.train_test_split(test_size=0.12, seed=42)
valid_test = split["test"].train_test_split(test_size=0.5, seed=42)

dataset = DatasetDict({
    "train": split["train"],
    "validation": valid_test["train"],
    "test": valid_test["test"]
})

print(f"\nSplit final:")
print(f"  Train: {len(dataset['train'])}")
print(f"  Val:   {len(dataset['validation'])}")
print(f"  Test:  {len(dataset['test'])}")

Dataset brut: 98345 exemples
Dataset doublé: 196690 exemples

Split final:
  Train: 173087
  Val:   11801
  Test:  11802


## 3. Tokenisation (NLLB spécifique)

In [5]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"
MAX_LENGTH = 128

print(f"Chargement tokenizer {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

def preprocess_function(examples, tokenizer, max_length=128):
    """Tokenisation avec gestion dynamique des codes langue NLLB"""
    inputs = [str(x).strip() if x else "" for x in examples["source"]]
    targets = [str(x).strip() if x else "" for x in examples["target"]]
    
    # Tokeniser inputs
    model_inputs = tokenizer(
        inputs,
        max_length=max_length,
        truncation=True,
        padding=False,
    )
    
    # Tokeniser targets avec codes langue NLLB
    tgt_codes = examples["tgt_lang_code"]
    labels_tokenized = tokenizer(
        targets,
        max_length=max_length,
        truncation=True,
        padding=False,
        add_special_tokens=False,
    )
    
    # Ajouter le token de langue au début de chaque label
    labels_list = []
    for label_ids, tgt_code in zip(labels_tokenized["input_ids"], tgt_codes):
        lang_id = tokenizer.convert_tokens_to_ids(tgt_code)
        new_labels = [lang_id] + list(label_ids) + [tokenizer.eos_token_id]
        labels_list.append(new_labels)
    
    model_inputs["labels"] = labels_list
    return model_inputs

print(f"Tokenisation en cours...")
tokenized_dataset = dataset.map(
    lambda ex: preprocess_function(ex, tokenizer, max_length=MAX_LENGTH),
    batched=True,
    batch_size=1000,
    remove_columns=["source", "target", "tgt_lang_code"],
    desc="Tokenisation NLLB"
)

print(f"✓ Tokenisation terminée")

Chargement tokenizer facebook/nllb-200-distilled-600M...
Tokenisation en cours...
✓ Tokenisation terminée


## 4. Configuration du modèle avec LoRA

In [13]:
# Nettoyage
import gc
gc.collect()
torch.cuda.empty_cache()

print(f"Chargement modèle {MODEL_NAME}...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Configuration LoRA pour NLLB
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# activer le gradient checkpointing Pour économiser la mémoire
# if hasattr(model, "enable_input_require_grads"):
#     model.enable_input_require_grads()
# else:
#     def make_inputs_require_grad(module, input, output):
#         output.requires_grad_(True)
#     model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)

# # Recommandé
# model.gradient_checkpointing_enable()

# Paramètres à logger
training_params = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": "q_proj,v_proj",
    "num_epochs": 4,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 8,
    "gradient_accumulation_steps": 2,
    "learning_rate": 1.2e-4,
    "warmup_steps": 500,
    "weight_decay": 0.01,
    "generation_max_length": 96,
    "generation_num_beams": 4,
    "dataset_size_train": len(tokenized_dataset["train"]),
    "dataset_size_val": len(tokenized_dataset["validation"]),
    "dataset_size_test": len(tokenized_dataset["test"]),
}

model.train()

Chargement modèle facebook/nllb-200-distilled-600M...
trainable params: 2,359,296 || all params: 617,433,088 || trainable%: 0.3821


PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): M2M100ForConditionalGeneration(
      (model): M2M100Model(
        (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
        (encoder): M2M100Encoder(
          (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
          (embed_positions): M2M100SinusoidalPositionalEmbedding()
          (layers): ModuleList(
            (0-11): 12 x M2M100EncoderLayer(
              (self_attn): M2M100Attention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1024, out_features=16, bias=False)
                  )
                  (lo

## 5. Définition des métriques

In [14]:
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def compute_metrics(eval_preds):
    """Calcul BLEU et chrF++"""
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Protection contre les IDs invalides
    max_id = len(tokenizer) - 1
    preds = np.clip(preds, 0, max_id)
    
    # Décodage des prédictions
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Préparation des labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.clip(labels, 0, max_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Nettoyage léger
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    # BLEU
    bleu_result = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[label] for label in decoded_labels],
        tokenize="13a"
    )

    # chrF++
    chrf_result = chrf_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,          # liste simple acceptée
        char_order=6,
        word_order=2,
        beta=2
    )

    return {
        "bleu": round(bleu_result["score"], 2),
        "chrf": round(chrf_result["score"], 2)
    }

print("✓ Métriques configurées")

✓ Métriques configurées


## 6. Entraînement avec MLflow

In [15]:
output_dir = "/kaggle/working/nllb-fr-wo-lora"
os.makedirs(output_dir, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    # overwrite_output_dir=True,
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1.2e-4,
    warmup_steps=500,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=96,
    generation_num_beams=4,
    report_to="none",  # On utilise MLflow directement via trainer
    push_to_hub=False,
    seed=42,
    logging_steps=50,
    save_total_limit=2
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if training_args.fp16 else None
)

# small_dataset = {
#     "train": tokenized_dataset["train"].select(range(200)),
#     "validation": tokenized_dataset["validation"].select(range(50)),
#     "test": tokenized_dataset["test"].select(range(50))
# }

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    # train_dataset=small_dataset["train"],
    # eval_dataset=small_dataset["validation"],
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("✓ Trainer configuré")

✓ Trainer configuré


## 7. Exécution de l'entraînement avec logging MLflow

In [16]:
# 1. Exécution de l'entraînement avec logging MLflow
from datetime import datetime
from transformers import pipeline

with mlflow.start_run(run_name=f"nllb-fr-wo-{datetime.now().strftime('%Y%m%d-%H%M')}") as run:
    run_id = run.info.run_id
    print(f"\n{'='*60}")
    print(f"Démarrage entraînement - Run ID: {run_id}")
    print(f"{'='*60}\n")

    # --- 1. Logger les hyperparamètres importants ---
    mlflow.log_params(training_params)

    # # (Optionnel) logger seulement quelques training_args utiles
    # useful_args = {
    #     "fp16": training_args.fp16,
    #     "eval_strategy": training_args.eval_strategy,
    #     "save_strategy": training_args.save_strategy,
    #     "generation_max_length": training_args.generation_max_length,
    #     "generation_num_beams": training_args.generation_num_beams,
    #     "metric_for_best_model": training_args.metric_for_best_model,
    # }
    # mlflow.log_params(useful_args)

    # --- 2. Entraînement ---
    train_result = trainer.train()

    # Logger les métriques d'entraînement
    for key, value in train_result.metrics.items():
        if isinstance(value, (int, float)):
            mlflow.log_metric(f"train_{key}", value)

    # --- 3. Évaluation sur le test ---
    print(f"\n{'='*60}")
    print("Évaluation sur le test set...")
    print(f"{'='*60}\n")

    test_results = trainer.evaluate(eval_dataset=small_dataset["test"])
    # test_results = trainer.evaluate(eval_dataset=tokenized_dataset["test"])

    for key, value in test_results.items():
        if isinstance(value, (int, float)):
            mlflow.log_metric(f"test_{key}", value)
            print(f"{key}: {value:.4f}")

    # --- 4. Sauvegarde des adapters LoRA (AVANT le merge) ---
    print(f"\nSauvegarde des adapters LoRA...")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    # Logger les adapters LoRA
    if os.path.exists(os.path.join(output_dir, "adapter_config.json")):
        mlflow.log_artifacts(output_dir, artifact_path="lora_adapters")

    # --- 5. Merge LoRA + création d'un modèle fusionné ---
    print("Fusion LoRA + modèle de base...")
    merged_model = model.merge_and_unload()

    # Sauvegarder le modèle fusionné séparément
    merged_dir = os.path.join(output_dir, "merged")
    os.makedirs(merged_dir, exist_ok=True)
    merged_model.save_pretrained(merged_dir)
    tokenizer.save_pretrained(merged_dir)

    # --- 6. Logger le modèle fusionné dans MLflow ---
    # Note : pour NLLB on log plutôt le modèle + tokenizer
    # plutôt qu'un pipeline "translation" générique
    try:
        mlflow.transformers.log_model(
            transformers_model={
                "model": merged_model,
                "tokenizer": tokenizer
            },
            artifact_path="nllb-merged-model",          # ou name= selon ta version MLflow
            task="translation",
        )
        print("✓ Modèle fusionné loggé dans MLflow")
    except Exception as e:
        print(f"⚠ mlflow.transformers.log_model a échoué : {e}")
        print("→ On log juste le dossier merged en artefact")
        mlflow.log_artifacts(merged_dir, artifact_path="nllb-merged-model")

    # --- 7. Résumé ---
    print(f"\n{'='*60}")
    print(f"✓ Entraînement terminé")
    print(f"✓ Run ID MLflow : {run_id}")
    print(f"✓ Adapters LoRA  : {output_dir}")
    print(f"✓ Modèle fusionné: {merged_dir}")
    print(f"{'='*60}")


Démarrage entraînement - Run ID: 7deb1938645547b680270eb2bcdbcb07



/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss


🏃 View run nllb-fr-wo-20261001-2354 at: https://dagshub.com/DrEPL/ml-translation-wo-fr.mlflow/#/experiments/0/runs/7deb1938645547b680270eb2bcdbcb07
🧪 View experiment at: https://dagshub.com/DrEPL/ml-translation-wo-fr.mlflow/#/experiments/0


KeyboardInterrupt: 

In [ ]:
import os
print(os.listdir(output_dir))
print(output_dir)

## 8. Enregistrement du modèle dans le Registry MLflow (optionnel)

In [12]:
# Optionnel: enregistrer le meilleur modèle dans le registry
run = mlflow.get_run(run_id)
model_uri = f"runs://{run_id}/nllb-model"

# try:
#     mlflow.register_model(
#         model_uri=model_uri,
#         name="nllb-fr-wo-translation",
#         tags={
#             "task": "machine_translation",
#             "source_lang": "french",
#             "target_lang": "wolof",
#             "bidirectional": "true",
#             "framework": "transformers",
#             "adapter": "lora"
#         }
#     )
#     print(f"✓ Modèle enregistré dans le Registry MLflow")
# except Exception as e:
#     print(f"Note: {e}")

print(f"\n{'='*60}")
print(f"FIN DU TRAINING - RÉSUMÉ")
print(f"{'='*60}")
print(f"BLEU Score (test): {test_results.get('eval_bleu', 'N/A')}")
print(f"chrF Score (test): {test_results.get('eval_chrf', 'N/A')}")
print(f"MLflow Run ID: {run_id}")
print(f"Model URI: {model_uri}")
print(f"{'='*60}")

Note: RESOURCE_DOES_NOT_EXIST: Run not found

FIN DU TRAINING - RÉSUMÉ
BLEU Score (test): 0.99
chrF Score (test): 9.91
MLflow Run ID: 9dcee913472343d78f28b3ea3cd3b0e2
Model URI: runs://9dcee913472343d78f28b3ea3cd3b0e2/nllb-model


Successfully registered model 'nllb-fr-wo-translation'.


In [ ]:
!zip -r nllb-fr-wo-lora.zip nllb-fr-wo-lora